# 17강 실습: 위키문헌 단편소설 코퍼스 만들기

오늘은 한국어 위키문헌의 `분류:단편소설` 187페이지를 내려받아 작은 코퍼스를 만든다. 이 분류의 글은 대부분 저작권 보호 기간이 끝난 작품이고, `robots.txt`와 위키미디어의 로봇 정책이 `/wiki/제목` 주소를 허용한다.

읽어 볼 물음은 이것이다. **12강의 온라인 대화에서 8,636번 혼자 쓰인 `완전`은 지난날의 소설에서 어떻게 쓰였을까?**

터미널에서 먼저 `pip install requests beautifulsoup4`와 `mkdir -p data/wikisource/raw corpus/wikisource`를 한다. 12강의 `data/nikl/utterances.csv`를 다시 쓴다.

**요청을 보내기 전에** 아래 `HEADERS`의 `your-id`를 자기 것으로 바꾼다. 서버 관리자가 누구의 요청인지 알 수 있도록 하는 줄이다.

## 0. 준비

`string_storage`는 16강의 그 줄이다. 오늘도 `\b`를 쓰는데, `datasets`와 함께 설치된 `pyarrow`가 있으면 이 줄 없이는 오류 없이 0이 나온다.

In [1]:
import os, re, time
from glob import glob
from urllib import robotparser
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
pd.set_option('mode.string_storage', 'python')


## 1. `robots.txt` 읽기

`robots.txt`는 사이트가 로봇에게 들어오지 말라고 적어 둔 경로의 목록이다. `requests`로 읽고 `robotparser`에 넘긴다.

- `blog.naver.com`은 `GPTBot` 같은 AI 봇을 전부 막는다(첫 줄 `False`).
- 둘째 줄의 `True`는 **허락받았다는 뜻이 아니다.** `robots.txt`는 로봇에게 하는 안내일 뿐이고, 약관과 저작권은 따로 있다. 블로그의 말을 연구하려면 모두의 말뭉치의 온라인 게시 자료 말뭉치 2025(네이버 블로그 73,176건)를 신청한다.

In [2]:
HEADERS = {'User-Agent': 'LDS2026-practice/0.1 (your-id@snu.ac.kr)'}
def read_robots(site):
    res = requests.get(site + '/robots.txt', headers=HEADERS, timeout=30)
    rp = robotparser.RobotFileParser()
    rp.parse(res.text.splitlines())
    return rp

naver = read_robots('https://blog.naver.com')
post = 'https://blog.naver.com/someone/123'
print(naver.can_fetch('GPTBot', post))
print(naver.can_fetch('LDS2026-practice', post))


False
True


위키문헌에서 `/wiki/제목`은 되고 `/w/`는 안 된다. `oldid`로 옛 버전을 받는 주소(둘째 줄)에는 로봇이 들어가지 않는다. 위키미디어의 로봇 정책도 쿼리 없는 `/wiki/제목`만 모으라고 한다.

In [3]:
BASE = 'https://ko.wikisource.org'
AGENT = 'LDS2026-practice'
wiki = read_robots(BASE)
print(wiki.can_fetch(AGENT, BASE + '/wiki/운수_좋은_날'))
print(wiki.can_fetch(AGENT, BASE + '/w/index.php?oldid=468919'))


True
False


## 2. 한 페이지 읽기

- `status_code`: 200은 성공, 404는 없음, 429는 너무 잦음.
- `fetch`: 같은 인자를 되풀이하지 않으려고 만든 작은 함수다.
- `headers`: 내가 누구인지 밝힌다.
- `timeout`: 응답이 없으면 30초 뒤 포기한다.
- `r.content`는 바이트, `r.text`는 글자다. 9강에서 본 차이다.

(2026-11-03에 받은 수다. 페이지가 고쳐지면 달라진다.)

In [4]:
def fetch(path):
    return requests.get(BASE + path, headers=HEADERS, timeout=30)
r = fetch('/wiki/운수_좋은_날')
print(r.status_code, r.headers['content-type'])
print(r.encoding, len(r.content), len(r.text))


200 text/html; charset=UTF-8
UTF-8 91501 73699


- `BeautifulSoup`: HTML을 14강의 XML처럼 나무로 만든다.
- `select_one`: CSS 선택자로 하나 찾는다. `div.mw-parser-output`은 `class`가 그 이름인 `div`다.
- `find_all`: 이름으로 모두 찾는다.

마지막 문단은 소설이 아니라 라이선스 상자의 글자다.

In [5]:
soup = BeautifulSoup(r.text, 'html.parser')
body = soup.select_one('div.mw-parser-output')
paras = [p.get_text().strip() for p in body.find_all('p')]
print(len(paras), paras[1][:30])
print(paras[-1][:30])


88 이날이야말로 동소문 안에서 인력거꾼 노릇을 하는 김 첨
Public domainPublic domainfals


라이선스 상자를 나무에서 잘라 낸다(`decompose`). 이 문구는 187페이지 모두에 있어서, 지우지 않았다면 약 5만 자가 소설로 세어진다. 코드가 오류 없이 돌았다고 맞는 것이 아니다. **첫 페이지와 끝 페이지를 읽어 본다.**

In [6]:
for box in body.select('.licenseContainer'):
    box.decompose()
paras = [p.get_text().strip() for p in body.find_all('p')]
paras = [p for p in paras if p]
print(len(paras), paras[-1][:30])


82 “설렁탕을 사다놓았는데 왜 먹지를 못하니, 왜 먹지를 


2026-11-03에 받은 버전의 첫 문단에는 그날 아침 누군가 넣은 장난 편집 아홉 줄이 있었다. 위키는 누구나 고친다. 여러분이 받을 때는 이미 사라졌을 수 있고, 그러면 이 셀의 결과가 달라진다. 페이지는 바뀐다.

In [7]:
lines = paras[0].split('\n')
print(len(lines), lines[1][:5])


10 안녕하세요


페이지의 소스 안에 버전 번호가 있다. **웹의 재현은 두 겹이다.** 버전 번호와 내려받은 날을 기록하고, 받은 HTML을 그대로 둔다. 모든 사이트가 버전 번호를 주지는 않는다. 그러면 날짜와 원본이 전부다.

In [8]:
print(re.search(r'"wgRevisionId":(\d+)', r.text).group(1))


468919


## 3. 여러 페이지 모으기

목록 페이지의 `#mw-pages` 구역에서 링크를 모은다. `#`은 `id`, `li a`는 `li` 안의 `a`다. `a['title']`처럼 속성은 사전처럼 꺼낸다.

In [9]:
page = BeautifulSoup(fetch('/wiki/분류:단편소설').text, 'html.parser')
links = [(a['title'], a['href']) for a in page.select('#mw-pages li a')]
print(len(links), links[1][0])


187 B사감과 러브레터


**3분 넘게 걸린다.** 돌려 놓고 다음 설명을 읽는다.

- `time.sleep(1)`: 한 번 받고 1초 쉰다. 수강생이 함께 돌리니 줄이지 않는다.
- `os.path.exists`: 이미 받은 페이지는 건너뛴다. 끊겨도 이어서 하고, 다시 돌려도 서버에 가지 않는다.
- `raise_for_status`: 200이 아니면 멈춘다. 오류 페이지를 본문인 척 저장하지 않는다.

429를 받으면 더 빨리 다시 시도하지 말고 응답의 `Retry-After`만큼 기다린다.

In [10]:
for title, href in tqdm(links):
    path = 'data/wikisource/raw/' + title.replace('/', '_') + '.html'
    if os.path.exists(path):
        continue
    res = fetch(href)
    res.raise_for_status()
    with open(path, 'w', encoding='utf-8') as f:
        f.write(res.text)
    time.sleep(1)


앞에서 한 일을 함수로 묶는다. 작가는 `title`이 `저자:`로 시작하는 첫 링크에서, 연도와 라이선스는 페이지 아래의 분류에서 꺼낸다.

In [11]:
def parse(path):
    with open(path, encoding='utf-8') as f:
        html = f.read()
    soup = BeautifulSoup(html, 'html.parser')
    body = soup.select_one('div.mw-parser-output')
    for box in body.select('.licenseContainer'):
        box.decompose()
    paras = [p.get_text().strip() for p in body.find_all('p')]
    cats = [a.get_text() for a in soup.select('#mw-normal-catlinks li a')]
    years = [c[:4] for c in cats if re.fullmatch(r'\d{4}년 작품', c)]
    return {'title': path.split('/')[-1].removesuffix('.html'),
            'author': body.select_one('a[title^="저자:"]').get_text(),
            'year': int(years[0]) if years else None,
            'license': ' '.join(c for c in cats if c.startswith('PD-')),
            'rev': int(re.search(r'"wgRevisionId":(\d+)', html).group(1)),
            'text': '\n'.join(p for p in paras if p)}


내려받은 파일만 읽는다. 서버에 가지 않는다. `astype('Int64')`는 빈칸을 품을 수 있는 정수다.

In [12]:
paths = sorted(glob('data/wikisource/raw/*.html'))
docs = pd.DataFrame([parse(p) for p in paths])
docs['year'] = docs['year'].astype('Int64')
print(docs.shape)
print(docs[['title', 'author', 'year', 'license', 'rev']].head(3))


(187, 6)
       title author  year    license     rev
0     B녀의 소묘    이무영  1934  PD-old-50  384422
1  B사감과 러브레터    현진건  1925  PD-old-50  464471
2         X씨    김동인  1925  PD-old-70  385642


## 4. 코퍼스 점검하기

넷 중 하나가 김동인이고 74페이지에는 연도가 없다. `김유정론`(작가론)과 번역 동화도 이 분류에 있다. 16강의 태그처럼 분류도 사람이 붙인다. 이 표는 당시 문단이 아니라 **입력한 사람의 선택**을 보여 준다.

In [13]:
print(docs['author'].value_counts().head(3).to_dict())
decades = pd.cut(docs['year'], [1900, 1919, 1929, 1939, 1959],
                 labels=['~1919', '1920s', '1930s', '1940~'])
print(decades.value_counts(sort=False).to_dict())
print(docs['year'].isna().sum())


{'김동인': 47, '김유정': 17, '강경애': 17}
{'~1919': 9, '1920s': 24, '1930s': 71, '1940~': 9}
74


번역 1편과 장난 편집을 지운다. 편집자 주석, 한자 병기, 옛 맞춤법, 이상의 붙여쓰기는 그대로 둔다. 고친 것과 고치지 않은 것을 모두 카드에 적는다.

`str.replace`는 장난 편집이 이미 사라졌으면 아무 일도 하지 않는다.

In [14]:
docs = docs[~docs['title'].str.startswith('번역:')].copy()
JUNK = r'안녕하세요 \S+입니다\n?'
docs['text'] = docs['text'].str.replace(JUNK, '', regex=True)
docs['chars'] = docs['text'].str.len()
print(len(docs), docs['chars'].sum())


186 2131049


## 5. 세어 보기

- `아주`: 문자열 362, `\b` 212. `아주머니`가 69번이다. `\b`가 막아 준다.
- `완전`: 문자열 71, **`\b` 0**. 186편에서 한 번도 혼자 쓰이지 않았다.
- `완전히`: 문자열 57, `\b` 51. 이상의 붙여쓰기(`완전히잊어버릴`)는 `\b`로 잡히지 않아 문자열로 센다.

In [15]:
text = docs['text']
pats = ['아주', r'\b아주\b', '아주머니',
        '완전', r'\b완전\b',
        '완전히', r'\b완전히\b']
for pat in pats:
    print(pat, text.str.count(pat).sum())


아주 362
\b아주\b 212
아주머니 69
완전 71
\b완전\b 0
완전히 57
\b완전히\b 51


크기가 다른 코퍼스는 비율로 비교한다(8강). `아주`는 비슷하고, `완전`과 `완전히`는 자리가 뒤집힌다.

**말할 수 있는 것:** 위키문헌의 이 단편소설들에서 `완전`이 혼자 쓰인 예는 없다. 김다미(2021)가 그린 시기 구분과 잘 맞는 수다.

**말할 수 없는 것:** '1930년대 사람들은 완전을 부사로 쓰지 않았다'는 아니다. 문체, 매체, 시기가 한꺼번에 다르므로 두 칸의 차이를 시기 하나로 돌릴 수 없다.

In [16]:
PATS = {'완전': r'\b완전\b', '완전히': '완전히', '아주': r'\b아주\b'}
def per_million(s):
    return {w: s.str.count(p).sum() / s.str.len().sum() * 1e6
            for w, p in PATS.items()}
u = pd.read_csv('data/nikl/utterances.csv')
table = pd.DataFrame({'단편소설': per_million(text),
                      '온라인 대화': per_million(u['form'])})
print(table.round(1))


     단편소설  온라인 대화
완전    0.0   217.1
완전히  26.7     4.5
아주   99.5    85.4


작품이 많은 다섯 작가 사이에서도 10만 자당 수가 아홉 배 가까이 차이 난다. 13강에서 사람마다 다시 세었던 것과 같다. 코퍼스의 평균 하나로 말하지 않는다.

In [17]:
docs['아주'] = text.str.count(r'\b아주\b')
by = docs.groupby('author')[['chars', '아주']].sum()
by['n'] = docs['author'].value_counts()
by['per_100k'] = (by['아주'] / by['chars'] * 1e5).round(1)
print(by[by['n'] >= 15].sort_values('per_100k', ascending=False))


         chars  아주   n  per_100k
author                          
김유정     174862  40  17      22.9
강경애     173370  24  17      13.8
이상      160473  22  15      13.7
김동인     515984  32  47       6.2
윤백남     189990   5  15       2.6


## 6. 남기기

다듬은 표를 저장하고 다시 읽어 확인한다. 받은 HTML은 `data/`에 있어 `git`에 잡히지 않는다. `corpus/wikisource/README.md`에 코퍼스 카드를 쓴다(슬라이드의 예시).

In [18]:
docs.to_csv('corpus/wikisource/docs.csv', index=False)
back = pd.read_csv('corpus/wikisource/docs.csv')
print(back.shape, back['text'].str.len().sum())


(186, 8) 2131049


## 7. 실습: 정도 부사

- `count`: 페이지마다 횟수를 더한다.
- `contains`: 페이지마다 있으면 `True`. 합이 곧 편 수다(8강의 문서 빈도).

여기서는 횟수와 편 수의 순서가 같다. 가장 잦은 단어의 용례를 다섯 개 읽고, 오늘날의 쓰임과 같은지 말해 보자.

In [19]:
words = ['아주', '매우', '몹시', '퍽', '대단히', '너무']
print({w: int(text.str.count(rf'\b{w}\b').sum()) for w in words})
print({w: int(text.str.contains(rf'\b{w}\b').sum()) for w in words})


{'아주': 212, '매우': 83, '몹시': 196, '퍽': 111, '대단히': 33, '너무': 221}
{'아주': 90, '매우': 49, '몹시': 86, '퍽': 51, '대단히': 28, '너무': 96}


## 출처 적기

한국어 위키문헌 기여자들, `분류:단편소설`. 페이지마다 `rev`로 버전을 밝힌다(`corpus/wikisource/docs.csv`).
국립국어원 온라인 대화 말뭉치 2021(11강의 `utterances.csv`). 김다미(2021), 《국어학》 97.

주소, 요청 간격, 파싱 규칙, 다듬은 내용, 버전이 이 노트북과 표에 있다. 웹은 바뀌므로 코드만으로 같은 표가 다시 나온다는 보장은 없다. 그래서 `docs.csv`도 함께 남긴다.